# 练习实验：Decision Trees

本练习将从零实现 Decision Tree，并用它判断蘑菇可食用还是有毒。

## 目录

- [1 - 软件包](#1)
- [2 - 问题描述](#2)
- [3 - 数据集](#3)
  - [3.1 编码后的数据集](#3.1)
- [4 - 回顾 Decision Tree](#4)
  - [4.1 计算 Entropy](#4.1)
    - [练习 1](#ex01)
  - [4.2 划分数据集](#4.2)
    - [练习 2](#ex02)
  - [4.3 计算 Information Gain](#4.3)
    - [练习 3](#ex03)
  - [4.4 选择最佳分裂特征](#4.4)
    - [练习 4](#ex04)
- [5 - 构建 Decision Tree](#5)


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除本 Notebook 中不参与评分的单元，也不要添加新单元。
>
> **通过本作业后**，如果想尝试修改不参与评分的代码，可以按照 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 软件包

首先运行下面的单元，导入本作业所需的软件包。

- [NumPy](https://www.numpy.org)：Python 中进行矩阵运算的基础库。
- [Matplotlib](https://matplotlib.org)：常用的 Python 绘图库。
- `utils.py`：包含本作业的辅助函数，无需修改。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *
from utils import *

%matplotlib inline

<a name="2"></a>
## 2 - 问题描述

假设你正在创办一家种植并销售野生蘑菇的公司。由于并非所有蘑菇都可食用，你希望根据外观属性判断它们可食用还是有毒。已有一些数据可用于完成这个分类任务。

**注意：本数据集仅用于教学演示，不能作为识别可食用蘑菇的指南。**

<a name="3"></a>
## 3 - 数据集

先查看收集到的数据：

|                                                     | 菌盖颜色 | 菌柄形状 | 单生 | 可食用 |
|:---------------------------------------------------:|:---------:|:-----------:|:--------:|:------:|
| <img src="images/0.png" alt="drawing" width="50"/> |   棕色   |   渐细  |    是   |    1   |
| <img src="images/1.png" alt="drawing" width="50"/> |   棕色   |  渐粗  |    是   |    1   |
| <img src="images/2.png" alt="drawing" width="50"/> |   棕色   |  渐粗  |    否    |    0   |
| <img src="images/3.png" alt="drawing" width="50"/> |   棕色   |  渐粗  |    否    |    0   |
| <img src="images/4.png" alt="drawing" width="50"/> |   棕色   |   渐细  |    是   |    1   |
| <img src="images/5.png" alt="drawing" width="50"/> |    红色    |   渐细  |    是   |    0   |
| <img src="images/6.png" alt="drawing" width="50"/> |    红色    |  渐粗  |    否    |    0   |
| <img src="images/7.png" alt="drawing" width="50"/> |   棕色   |  渐粗  |    是   |    1   |
| <img src="images/8.png" alt="drawing" width="50"/> |    红色    |   渐细  |    否    |    1   |
| <img src="images/9.png" alt="drawing" width="50"/> |   棕色   |  渐粗  |    否    |    0   |

共有 10 个蘑菇样本，每个样本包含三个特征和一个标签：

- Cap Color：菌盖颜色，`Brown` 或 `Red`。
- Stalk Shape：菌柄形状，`Tapering`（形如 `\/`）或 `Enlarging`（形如 `/\`）。
- Solitary：是否单生，`Yes` 或 `No`。
- Edible：是否可食用，1 表示可食用，0 表示有毒。

<a name="3.1"></a>
### 3.1 编码后的数据集

为方便实现，将各个二元 Categorical Feature 编码为 0/1：

|                                                    | 棕色菌盖 | 渐细菌柄 | 单生 | 可食用 |
|:--------------------------------------------------:|:---------:|:--------------------:|:--------:|:------:|
| <img src="images/0.png" alt="drawing" width="50"/> |     1     |           1          |     1    |    1   |
| <img src="images/1.png" alt="drawing" width="50"/> |     1     |           0          |     1    |    1   |
| <img src="images/2.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |
| <img src="images/3.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |
| <img src="images/4.png" alt="drawing" width="50"/> |     1     |           1          |     1    |    1   |
| <img src="images/5.png" alt="drawing" width="50"/> |     0     |           1          |     1    |    0   |
| <img src="images/6.png" alt="drawing" width="50"/> |     0     |           0          |     0    |    0   |
| <img src="images/7.png" alt="drawing" width="50"/> |     1     |           0          |     1    |    1   |
| <img src="images/8.png" alt="drawing" width="50"/> |     0     |           1          |     0    |    1   |
| <img src="images/9.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |

因此，`X_train` 的每个样本包含三个特征：棕色菌盖为 1、红色为 0；渐细菌柄为 1、渐粗为 0；单生为 1、非单生为 0。

`y_train` 是目标标签：`y=1` 表示可食用，`y=0` 表示有毒。


In [ ]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### 查看变量

先打印变量，熟悉数据集。下面输出 `X_train` 的前几个元素及其变量类型。


In [ ]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

下面对 `y_train` 做同样的检查。


In [ ]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

#### 检查变量维度

查看维度也有助于理解数据。打印 `X_train` 和 `y_train` 的形状，确认训练样本的数量。


In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

<a name="4"></a>
## 4 - 回顾 Decision Tree

本实验将根据给定数据集构建 Decision Tree，步骤如下：

1. 将所有样本放入 Root Node。
2. 计算按每个候选特征分裂时的 Information Gain，选择最大的一个。
3. 按所选特征划分数据，创建左右分支。
4. 重复分裂过程，直到满足停止条件。

为此，你将实现以下函数：计算节点的 Entropy；按给定特征划分左右分支；计算该分裂的 Information Gain；选择使 Information Gain 最大的特征。

随后利用这些函数递归构建树。本实验采用的停止条件是最大深度为 2。


<a name="4.1"></a>
### 4.1 计算 Entropy

首先实现辅助函数 `compute_entropy`，计算节点的 Entropy，用于衡量 Impurity（不纯度）。输入 NumPy 数组 `y` 表示节点中各样本是否可食用：1 为可食用，0 为有毒。

先计算可食用样本的比例 $p_1$，再计算：

$$H(p_1)=-p_1\log_2(p_1)-(1-p_1)\log_2(1-p_1).$$

注意：

- 对数以 2 为底。
- 按约定 $0\log_2 0=0$。因此 `p1` 为 0 或 1 时，Entropy 直接设为 0。
- 先检查节点是否为空；若 `len(y)==0`，返回 0。

<a name="ex01"></a>
### 练习 1

根据以上说明完成 `compute_entropy()`。遇到困难时，可以查看下方提示。


In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # You need to return the following variables correctly
    entropy = 0.
    
    ### START CODE HERE ###
    
        
        
            
        
            
    ### END CODE HERE ###        
    
    return entropy

<details>
<summary><strong>点击查看提示</strong></summary>

- 通过 `y[y == 1]` 取得标签为 1 的样本，`len(y)` 返回总样本数。
- [`np.log2`](https://numpy.org/doc/stable/reference/generated/numpy.log2.html) 计算以 2 为底的对数。
- `p1` 为 0 或 1 时，将 Entropy 设为 0。

<details>
<summary><strong>点击查看更多提示</strong></summary>

可以采用以下函数结构：

```python
def compute_entropy(y):

    # You need to return the following variables correctly
    entropy = 0.

    ### START CODE HERE ###
    if len(y) != 0:
        # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
        p1 =

        # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
        if p1 != 0 and p1 != 1:
            # Your code here to calculate the entropy using the formula provided above
            entropy = 
        else:
            entropy = 0. 
    ### END CODE HERE ###        

    return entropy
```

<details>
<summary><strong>计算 p1 的提示</strong></summary>

`p1 = len(y[y == 1]) / len(y)`

</details>

<details>
<summary><strong>计算 Entropy 的提示</strong></summary>

`entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)`

</details>
</details>
</details>


运行下面的测试代码，检查你的实现是否正确：


In [ ]:
# Compute entropy at the root node (i.e. with all examples)
# Since we have 5 edible and 5 non-edible mushrooms, the entropy should be 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
compute_entropy_test(compute_entropy)

**预期输出：** Root Node 的 Entropy 为 `1.0`。


<a name="4.2"></a>
### 4.2 划分数据集

接下来实现 `split_dataset`。它接收训练数据、当前节点的样本索引列表和用于分裂的特征，并返回左右分支的样本索引。稍后再实现评估分裂质量的代码。

例如，从包含全部样本的 Root Node 开始，`node_indices=[0,1,2,3,4,5,6,7,8,9]`。选择特征 0（是否为棕色菌盖）分裂后，左分支为 `left_indices=[0,1,2,3,4,7,9]`，右分支为 `right_indices=[5,6,8]`。

|       |                                                    | 棕色菌盖 | 渐细菌柄 | 单生 | 可食用 |
|-------|:--------------------------------------------------:|:---------:|:--------------------:|:--------:|:------:|
| 0     | <img src="images/0.png" alt="drawing" width="50"/> |     1     |           1          |     1    |    1   |
| 1     | <img src="images/1.png" alt="drawing" width="50"/> |     1     |           0          |     1    |    1   |
| 2     | <img src="images/2.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |
| 3     | <img src="images/3.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |
| 4     | <img src="images/4.png" alt="drawing" width="50"/> |     1     |           1          |     1    |    1   |
| 5     | <img src="images/5.png" alt="drawing" width="50"/> |     0     |           1          |     1    |    0   |
| 6     | <img src="images/6.png" alt="drawing" width="50"/> |     0     |           0          |     0    |    0   |
| 7     | <img src="images/7.png" alt="drawing" width="50"/> |     1     |           0          |     1    |    1   |
| 8     | <img src="images/8.png" alt="drawing" width="50"/> |     0     |           1          |     0    |    1   |
| 9     | <img src="images/9.png" alt="drawing" width="50"/> |     1     |           0          |     0    |    0   |

<a name="ex02"></a>
### 练习 2

完成 `split_dataset()`。遍历 `node_indices` 中的每个索引：如果该样本在指定特征上的值为 1，就加入 `left_indices`；如果为 0，就加入 `right_indices`。遇到困难时，可以查看下方提示。


In [ ]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (list):     List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (list):     Indices with feature value == 1
        right_indices (list):    Indices with feature value == 0
    """
    
    # You need to return the following variables correctly
    left_indices = []
    right_indices = []
    
    ### START CODE HERE ###
    
        
            
        
            
    ### END CODE HERE ###
        
    return left_indices, right_indices

<details>
<summary><strong>点击查看提示</strong></summary>

可以采用以下函数结构：

```python
def split_dataset(X, node_indices, feature):
    left_indices = []
    right_indices = []

    ### START CODE HERE ###
    for i in node_indices:
        if ...:  # Check whether this feature has value 1
            left_indices.append(i)
        else:
            right_indices.append(i)
    ### END CODE HERE ###

    return left_indices, right_indices
```

<details>
<summary><strong>点击查看更多提示</strong></summary>

判断条件为 `if X[i][feature] == 1:`。

</details>
</details>


运行下面的单元检查实现。按之前讨论的方式，在 Root Node 上使用特征 0（Brown Cap）分裂全部样本。还提供了一个辅助函数，用于可视化分裂结果。


In [ ]:
# Case 1

root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Feel free to play around with these variables
# The dataset only has three features, so this value can be 0 (Brown Cap), 1 (Tapering Stalk Shape) or 2 (Solitary)
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("CASE 1:")
print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# Visualize the split 
generate_split_viz(root_indices, left_indices, right_indices, feature)

print()

# Case 2

root_indices_subset = [0, 2, 4, 6, 8]
left_indices, right_indices = split_dataset(X_train, root_indices_subset, feature)

print("CASE 2:")
print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# Visualize the split 
generate_split_viz(root_indices_subset, left_indices, right_indices, feature)

# UNIT TESTS    
split_dataset_test(split_dataset)

**预期输出**

```
CASE 1:
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]

CASE 2:
Left indices:  [0, 2, 4]
Right indices:  [6, 8]
```


<a name="4.3"></a>
### 4.3 计算 Information Gain

接下来实现 `compute_information_gain`，输入训练数据、节点中的样本索引和分裂特征，返回该分裂的 Information Gain。

<a name="ex03"></a>
### 练习 3

完成 `compute_information_gain()`，计算：

$$\text{Information Gain}=H(p_1^{\text{node}})-\left(w^{\text{left}}H(p_1^{\text{left}})+w^{\text{right}}H(p_1^{\text{right}})\right).$$

其中，$H(p_1^{\text{node}})$ 为当前节点的 Entropy，$H(p_1^{\text{left}})$ 和 $H(p_1^{\text{right}})$ 为左右子节点的 Entropy，$w^{\text{left}}$ 和 $w^{\text{right}}$ 为各子节点样本数占当前节点样本数的比例。

可调用前面实现的 `compute_entropy()` 计算 Entropy。起始代码已经调用 `split_dataset()` 划分数据。遇到困难时，可以查看下方提示。


In [ ]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        feature (int):           Index of feature to split on
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # You need to return the following variables correctly
    information_gain = 0
    
    ### START CODE HERE ###
    
    
    
    
    
    
    
    
    
    
    
    
    
    ### END CODE HERE ###  
    
    return information_gain

<details>
<summary><strong>点击查看提示</strong></summary>

可以采用以下函数结构：

```python
def compute_information_gain(X, y, node_indices, feature):
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)

    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    # You need to return the following variables correctly
    information_gain = 0

    ### START CODE HERE ###
    # Your code here to compute the entropy at the node using compute_entropy()
    node_entropy = 
    # Your code here to compute the entropy at the left branch
    left_entropy = 
    # Your code here to compute the entropy at the right branch
    right_entropy = 

    # Your code here to compute the proportion of examples at the left branch
    w_left = 

    # Your code here to compute the proportion of examples at the right branch
    w_right = 

    # Your code here to compute weighted entropy from the split using 
    # w_left, w_right, left_entropy and right_entropy
    weighted_entropy = 

    # Your code here to compute the information gain as the entropy at the node
    # minus the weighted entropy
    information_gain = 
    ### END CODE HERE ###  

    return information_gain
```

<details>
<summary><strong>计算各节点 Entropy 的提示</strong></summary>

```python
node_entropy = compute_entropy(y_node)
left_entropy = compute_entropy(y_left)
right_entropy = compute_entropy(y_right)
```

</details>

<details>
<summary><strong>计算 w_left 和 w_right 的提示</strong></summary>

```python
w_left = len(X_left) / len(X_node)
w_right = len(X_right) / len(X_node)
```

</details>

<details>
<summary><strong>计算 Weighted Entropy 的提示</strong></summary>

`weighted_entropy = w_left * left_entropy + w_right * right_entropy`

</details>

<details>
<summary><strong>计算 Information Gain 的提示</strong></summary>

`information_gain = node_entropy - weighted_entropy`

</details>
</details>


运行下面的单元检查实现，并计算按各特征分裂时的 Information Gain。


In [ ]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)

info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

**预期输出**

```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```


在 Root Node 上，按 Solitary（`feature=2`）分裂产生的 Information Gain 最大，因此它是最佳分裂特征。


<a name="4.4"></a>
### 4.4 选择最佳分裂特征

下面编写函数，逐个计算特征的 Information Gain，返回使其最大的特征。

<a name="ex04"></a>
### 练习 4

完成 `get_best_split()`。输入训练数据及当前节点的样本索引，输出最佳特征的索引。可以遍历所有特征，并调用 `compute_information_gain()` 计算对应值。遇到困难时，可以查看下方提示。


In [ ]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # Some useful variables
    num_features = X.shape[1]
    
    # You need to return the following variables correctly
    best_feature = -1
    
    ### START CODE HERE ###
    
    
    
        
        
            
            
    ### END CODE HERE ##    
   
    return best_feature

<details>
<summary><strong>点击查看提示</strong></summary>

可以采用以下函数结构：

```python
def get_best_split(X, y, node_indices):
    num_features = X.shape[1]
    best_feature = -1

    ### START CODE HERE ###
    max_info_gain = 0
    for feature in range(num_features):
        info_gain =  # Calculate the information gain for this feature
        if info_gain > max_info_gain:
            max_info_gain =  # Update the maximum
            best_feature =  # Update the best feature
    ### END CODE HERE ###

    return best_feature
```

<details>
<summary><strong>计算 info_gain 的提示</strong></summary>

`info_gain = compute_information_gain(X, y, node_indices, feature)`

</details>

<details>
<summary><strong>更新最大值与最佳特征的提示</strong></summary>

```python
max_info_gain = info_gain
best_feature = feature
```

</details>
</details>


运行下面的单元检查函数实现。


In [ ]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

与前面的结果一致，函数返回 Root Node 的最佳分裂特征为 2，即 Solitary。


<a name="5"></a>
## 5 - 构建 Decision Tree

本节使用前面实现的函数，持续选择最佳特征进行分裂，直到达到停止条件（最大深度为 2）。这一部分无需编写代码。


In [ ]:
# Not graded
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Maximum depth reached - stop splitting
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Otherwise, get best split and split the data
    # Get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices) 
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)

In [ ]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)
generate_tree_viz(root_indices, y_train, tree)

<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
